In [1]:
import pandas as pd

# ── LOAD ALL THREE FILES ──────────────────────────────────────────────────────
market_df = pd.read_csv("market_data.csv",
                        index_col="Date", parse_dates=True)

macro_df  = pd.read_csv("macro_data.csv",
                        index_col="Date", parse_dates=True)

events_df = pd.read_csv("policy_events.csv",
                        parse_dates=["Date"])

print(f"Market data:  {len(market_df)} rows, {len(market_df.columns)} columns")
print(f"Macro data:   {len(macro_df)} rows, {len(macro_df.columns)} columns")
print(f"Policy events: {len(events_df)} events")

# ── STEP 1: JOIN MARKET AND MACRO DATA BY DATE ────────────────────────────────
combined = market_df.join(macro_df, how="left")

# ── STEP 2: REMOVE WEEKENDS BEFORE FILLING ────────────────────────────────────
combined = combined[combined.index.dayofweek < 5]

# ── STEP 3: FORWARD FILL MONTHLY MACRO VALUES TO DAILY ───────────────────────
combined = combined.ffill()

# ── STEP 4: ADD POLICY EVENT COLUMNS ─────────────────────────────────────────
events_df = events_df.set_index("Date")

combined["POLICY_EVENT"]       = combined.index.isin(
                                     events_df.index).astype(int)
combined["POLICY_TYPE"]        = events_df["Event_Type"].reindex(
                                     combined.index)
combined["POLICY_DESCRIPTION"] = events_df["Description"].reindex(
                                     combined.index)
combined["POLICY_SEVERITY"]    = events_df["Severity"].reindex(
                                     combined.index)
combined["POLICY_ASSETS"]      = events_df["Assets_Affected"].reindex(
                                     combined.index)
combined["POLICY_PERIOD"]      = events_df["Period"].reindex(
                                     combined.index)

# ── STEP 5: REMOVE ROWS WHERE CORE MARKET PRICES ARE MISSING ─────────────────
combined = combined.dropna(subset=[
    "EURUSD_Close", "GBPUSD_Close", "GOLD_Close", "OIL_Close"
])

# ── STEP 6: ADD TRAIN/TEST SPLIT COLUMN ──────────────────────────────────────
combined["PERIOD"] = combined.index.map(
    lambda d: "TEST" if d >= pd.Timestamp("2025-01-01") else "TRAIN"
)

# ── STEP 7: SAVE FINAL FILE ───────────────────────────────────────────────────
combined.to_csv("project8_dataset.csv")

# ── SUMMARY REPORT ────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("COMBINED DATASET SAVED")
print("=" * 55)
print(f"File:          project8_dataset.csv")
print(f"Total rows:    {len(combined)}")
print(f"Total columns: {len(combined.columns)}")
print(f"Date range:    {combined.index.min().date()} "
      f"to {combined.index.max().date()}")

train = combined[combined["PERIOD"] == "TRAIN"]
test  = combined[combined["PERIOD"] == "TEST"]
print(f"\nTraining rows: {len(train)} (2018-2024)")
print(f"Test rows:     {len(test)}  (2025-2026)")

print(f"\nPolicy event days total: "
      f"{combined['POLICY_EVENT'].sum()}")
print(f"Policy event days train: "
      f"{train['POLICY_EVENT'].sum()}")
print(f"Policy event days test:  "
      f"{test['POLICY_EVENT'].sum()}")

print("\n--- All columns ---")
for col in combined.columns:
    print(f"  {col}")

print("\n--- Missing values per column ---")
missing = combined.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print("  No missing values")
else:
    print(missing.to_string())

print("\n--- First 3 rows ---")
print(combined.head(3).to_string())

print("\n--- Sample policy event day ---")
event_day = combined[combined["POLICY_EVENT"] == 1].iloc[0]
print(f"Date:        {event_day.name.date()}")
print(f"Type:        {event_day['POLICY_TYPE']}")
print(f"Severity:    {event_day['POLICY_SEVERITY']}")
print(f"Period:      {event_day['POLICY_PERIOD']}")
print(f"Description: {str(event_day['POLICY_DESCRIPTION'])[:100]}...")

ValueError: Missing column provided to 'parse_dates': 'Date'